In [ ]:
%load_ext autoreload
%autoreload 2         

import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # let the notebook see the repo root
from src.ingestion.eia_client import fetch_page

load_dotenv("../.env")
key = os.getenv("EIA_API_KEY")
print(len(key))

In [ ]:
url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

params = {
    "api_key": key,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "facets[type][]": "D",
    "start": "2026-09-01T00",
    "end": "2026-09-01T23",
}

In [ ]:
rows, total = fetch_page(url, params)
print(len(rows), total)     # want: 24 24

In [ ]:
bad_params = params.copy()
bad_params["facets[respondent][]"] = "CISOX"

rows, total = fetch_page(url, bad_params)
print(len(rows), total)

In [ ]:
from src.ingestion.eia_client import fetch_all

month = params.copy()
month["start"] = "2026-09-01T00"
month["end"] = "2026-09-30T23"

rows = fetch_all(url, month, page_size=200)
print(len(rows))

In [ ]:
from src.ingestion.raw_storage import save_raw
import json

rows = fetch_all(url, month)
path = save_raw(rows, "demand", month["start"], month["end"])
print(path)

with open(path) as f:
    reloaded = json.load(f)          # read it back: the opposite of ④
print(len(reloaded) == len(rows))       # want: True

In [13]:
from src.ingestion.backfill import backfill

backfill("demand", url, params, first="2026-08-01")

demand 2026-08: exists, skipping
demand 2026-09: exists, skipping
Fetched 694 / 694
demand 2026-10: saved 694 rows
